# Analysis
Network analysis of Bluesky discourse around machine translation.

**Research questions:**
1. **Sentiment & framing** — Do people talk about MT as a tool, a threat, or both?
2. **Professional impact discourse** — How do working translators discuss MT's effect on their careers?
3. **Utility vs. displacement tension** — Do people acknowledge MT's utility while lamenting professional erosion?

## Setup

In [ ]:
!pip -q install networkx matplotlib pandas

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.patches import Patch
import numpy as np
from collections import Counter

# Update this path if running on Colab with Drive mounted
DATA_DIR = 'data/'

In [ ]:
df = pd.read_csv(DATA_DIR + 'posts_clean.csv')
df_profiles = pd.read_csv(DATA_DIR + 'author_profiles.csv')
df_graph = pd.read_csv(DATA_DIR + 'graph.csv')

print(f"Posts:    {len(df)}")
print(f"Profiles: {len(df_profiles)}")
print(f"Edges:    {len(df_graph)} ({df_graph['type'].value_counts().to_dict()})")

## Network construction

In [ ]:
# Build directed graph
G = nx.from_pandas_edgelist(
    df_graph, source='source', target='target',
    edge_attr='type', create_using=nx.DiGraph()
)

# Attach node attributes
user_type_map = dict(zip(df['handle'], df['user_type']))
post_count_map = df['handle'].value_counts().to_dict()

for node in G.nodes():
    G.nodes[node]['user_type'] = user_type_map.get(node, 'unknown')
    G.nodes[node]['post_count'] = post_count_map.get(node, 0)
    G.nodes[node]['in_posts'] = node in user_type_map

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

## Basic graph statistics

In [ ]:
print("=" * 45)
print("FULL GRAPH")
print("=" * 45)
print(f"Nodes:   {G.number_of_nodes()}")
print(f"Edges:   {G.number_of_edges()}")
print(f"Density: {nx.density(G):.6f}")

# Subgraph: only nodes that appear as post authors
post_authors = set(df['handle'])
G_posts = G.subgraph(post_authors).copy()

print(f"\n{'='*45}")
print("POST AUTHORS SUBGRAPH")
print("=" * 45)
print(f"Nodes:   {G_posts.number_of_nodes()}")
print(f"Edges:   {G_posts.number_of_edges()}")
print(f"Density: {nx.density(G_posts):.4f}")

wcc = list(nx.weakly_connected_components(G_posts))
print(f"Weakly connected components: {len(wcc)}")
print(f"Largest component size:      {max(len(c) for c in wcc)}")

## Centrality

In [ ]:
# In-degree: who gets followed / replied to most
in_degree = dict(G.in_degree())
top_in = sorted(in_degree.items(), key=lambda x: x[1], reverse=True)[:15]

print("Top 15 by in-degree (most followed/replied-to):")
print(f"  {'handle':<42} {'in-deg':>7}  type")
print("  " + "-" * 60)
for handle, deg in top_in:
    utype = user_type_map.get(handle, 'unknown')
    print(f"  {handle:<42} {deg:>7}  {utype}")

In [ ]:
# Reply subgraph centrality
reply_edges = df_graph[df_graph['type'] == 'reply']
G_reply = nx.from_pandas_edgelist(
    reply_edges, source='source', target='target',
    create_using=nx.DiGraph()
)

in_degree_reply = dict(G_reply.in_degree())
top_reply = sorted(in_degree_reply.items(), key=lambda x: x[1], reverse=True)[:10]

print(f"Reply network — nodes: {G_reply.number_of_nodes()}, edges: {G_reply.number_of_edges()}")
print(f"\nTop 10 most replied-to users:")
print(f"  {'handle':<42} {'replies':>7}  type")
print("  " + "-" * 60)
for handle, deg in top_reply:
    utype = user_type_map.get(handle, 'unknown')
    print(f"  {handle:<42} {deg:>7}  {utype}")

In [ ]:
# Betweenness centrality on post-authors subgraph (bridges between communities)
betweenness = nx.betweenness_centrality(G_posts, normalized=True)
top_between = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]

print("Top 10 by betweenness centrality (bridges between groups):")
print(f"  {'handle':<42} {'between':>8}  type")
print("  " + "-" * 62)
for handle, score in top_between:
    utype = user_type_map.get(handle, 'unknown')
    print(f"  {handle:<42} {score:>8.4f}  {utype}")

## Community detection

In [ ]:
# Louvain on undirected post-authors subgraph
G_posts_undirected = G_posts.to_undirected()

# Remove isolates (no edges to anyone in the posts subgraph)
isolates = list(nx.isolates(G_posts_undirected))
G_posts_connected = G_posts_undirected.copy()
G_posts_connected.remove_nodes_from(isolates)
print(f"Nodes after removing isolates: {G_posts_connected.number_of_nodes()}")

communities = nx.community.louvain_communities(G_posts_connected, seed=42)
communities = sorted(communities, key=len, reverse=True)

print(f"Communities found: {len(communities)}")
print()
for i, c in enumerate(communities[:8]):
    types = Counter(user_type_map.get(n, 'unknown') for n in c)
    print(f"  Community {i+1:>2}: {len(c):>4} nodes — {dict(types)}")

In [ ]:
# Assign community ID to posts and inspect each community
node_community = {}
for i, community in enumerate(communities):
    for node in community:
        node_community[node] = i

df['community'] = df['handle'].map(node_community)

print("User type breakdown + sample posts per community (top 5):\n")
for i in range(min(5, len(communities))):
    comm_df = df[df['community'] == i]
    if comm_df.empty:
        continue
    print(f"── Community {i+1} ({len(communities[i])} nodes, {len(comm_df)} posts) ──")
    print(comm_df['user_type'].value_counts().to_string())
    print("Sample posts:")
    for t in comm_df['text_clean'].sample(min(3, len(comm_df)), random_state=42).tolist():
        print(f"  • {t[:110]}")
    print()

## Visualizations

In [ ]:
# Reply network colored by user type
color_map = {
    'professional': '#e63946',
    'tech':         '#2a9d8f',
    'general':      '#adb5bd',
    'unknown':      '#dee2e6',
}

pos = nx.spring_layout(G_reply, seed=42, k=0.6)
node_colors = [color_map.get(user_type_map.get(n, 'unknown'), '#dee2e6') for n in G_reply.nodes()]
node_sizes  = [80 + in_degree_reply.get(n, 0) * 60 for n in G_reply.nodes()]

fig, ax = plt.subplots(figsize=(12, 9))
nx.draw_networkx_nodes(G_reply, pos, node_color=node_colors, node_size=node_sizes, alpha=0.85, ax=ax)
nx.draw_networkx_edges(G_reply, pos, alpha=0.2, arrows=True, arrowsize=8, ax=ax)

top_handles = {h for h, _ in top_reply[:6]}
labels = {n: n.split('.')[0] for n in G_reply.nodes() if n in top_handles}
nx.draw_networkx_labels(G_reply, pos, labels=labels, font_size=7, ax=ax)

legend_patches = [Patch(color=c, label=t) for t, c in color_map.items() if t != 'unknown']
ax.legend(handles=legend_patches, loc='upper left', fontsize=9)
ax.set_title("Reply Network — node size = replies received", fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.savefig(DATA_DIR + 'reply_network.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Community graph: post-authors colored by community
community_colors = plt.cm.tab10.colors
node_comm_colors = [
    community_colors[node_community.get(n, len(community_colors) - 1) % len(community_colors)]
    for n in G_posts_connected.nodes()
]

pos_c = nx.spring_layout(G_posts_connected, seed=42, k=0.4)
fig, ax = plt.subplots(figsize=(12, 9))
nx.draw_networkx_nodes(G_posts_connected, pos_c, node_color=node_comm_colors, node_size=60, alpha=0.8, ax=ax)
nx.draw_networkx_edges(G_posts_connected, pos_c, alpha=0.1, ax=ax)

legend_patches = [
    Patch(color=community_colors[i % len(community_colors)], label=f"Community {i+1} (n={len(communities[i])})")
    for i in range(min(8, len(communities)))
]
ax.legend(handles=legend_patches, loc='upper left', fontsize=8)
ax.set_title("Post-author communities (Louvain)", fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.savefig(DATA_DIR + 'communities.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# In-degree distribution
in_degrees = [d for _, d in G.in_degree() if d > 0]

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(in_degrees, bins=40, color='#457b9d', edgecolor='white', log=True)
ax.set_xlabel("In-degree")
ax.set_ylabel("Count (log scale)")
ax.set_title("In-degree Distribution (full graph)")
plt.tight_layout()
plt.savefig(DATA_DIR + 'degree_distribution.png', dpi=150, bbox_inches='tight')
plt.show()